# Introduction to Lakehouse Runtime Catalog for Iceberg

This notebook showcases the Lakehouse runtime catalog for Apache Iceberg with a minimum viable and practical example of implementing medallion architecture with serverless interactive Managed Spark service as the lakehouse engine.

## 1. Setup

Configure environment variables. Provide your project ID and a [region](https://cloud.google.com/bigquery/docs/locations#regions) to store your resources, such as `us-central1`. Also create a serverless interactive Spark session.

In [ ]:
PROJECT_ID_LIST=!gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID_LIST[0]
PROJECT_NBR= ! gcloud projects describe $PROJECT_ID | grep projectNumber | cut -d':' -f2 | xargs
PROJECT_NBR=PROJECT_NBR[0]
LOCATION = "us-central1"
STAGE_BUCKET_NAME = f"froyo-lakehouse-staging-{PROJECT_NBR}"
ICEBERG_LAKEHOUSE_BUCKET_NAME = f"froyo_iceberg_lakehouse_catalog_{PROJECT_NBR}"
ICEBERG_CATALOG_NAME="froyo_iceberg_catalog"
ICEBERG_NAMESPACE="froyo_ns"
APP_NAME="froyo_app"

Create a spark session with Iceberg catalog configuration

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session
from pyspark.sql import functions as F

# Create the Dataproc Serverless session.
s8s_spark_session = Session()

# Serverless runtime at authoring was 3.0 with Iceberg 1.10
s8s_spark_session.runtime_config.properties[f"spark.sql.defaultCatalog"] = ICEBERG_CATALOG_NAME
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}"] = "org.apache.iceberg.spark.SparkCatalog"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.type"] = "rest"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.uri"] = "https://biglake.googleapis.com/iceberg/v1/restcatalog"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.warehouse"] = f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.io-impl"] = "org.apache.iceberg.gcp.gcs.GCSFileIO"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.header.x-goog-user-project"] = PROJECT_ID
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.rest.auth.type"] = "org.apache.iceberg.gcp.auth.GoogleAuthManager"
s8s_spark_session.runtime_config.properties[f"spark.sql.extensions"] = "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.rest-metrics-reporting-enabled"] = "false"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.quickstart_catalog.header.X-Iceberg-Access-Delegation"] = "vended-credentials"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.quickstart_catalog.gcs.oauth2.refresh-credentials-endpoint"] = "https://oauth2.googleapis.com/token"

spark = (DataprocSparkSession.builder
    .appName(APP_NAME)
    .dataprocSessionConfig(s8s_spark_session)
    .getOrCreate())

## 2. Create an Iceberg namespace

In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS froyo_ns;")
spark.sql(f"USE froyo_ns;")

In [ ]:
spark.sql("SHOW NAMESPACES;").show(truncate=False)

In [ ]:
spark.sql("SHOW TABLES IN froyo_ns").show(truncate=False)

In [ ]:
# Drop any existing tables in case of a rerun

# 1. Get the dataframe containing the tables
tables_df = spark.sql("SHOW TABLES IN froyo_ns")

# 2. Collect rows to the driver
# (SHOW TABLES outputs columns: 'namespace', 'tableName', and 'isTemporary')
tables_list = tables_df.collect()

print(f"Found {len(tables_list)} targets in 'froyo_ns'. Starting iterative drop...")

# 3. Loop and drop
for row in tables_list:
    table_name = row['tableName']
    is_temp = row['isTemporary']

    # Fully qualify the name to ensure you drop from the correct namespace
    fq_name = f"froyo_ns.{table_name}"

    try:
        if is_temp:
            # If it's a temporary view, use DROP VIEW
            print(f"Dropping temporary view: {table_name}")
            spark.sql(f"DROP TEMPORARY VIEW IF EXISTS {table_name}")
        else:
            # Standard or Iceberg table
            print(f"Dropping table: {fq_name}")
            spark.sql(f"DROP TABLE IF EXISTS {fq_name}")

    except Exception as e:
        print(f"⚠️ Failed to drop {table_name}: {e}")

print("Iterative drop process complete.")

# 3. [Medallion Architecture] Build the raw layer / bronze layer
Staging layers have transient data. Its a best practice to copy into bronze layer as is - to stay true to source.

In [ ]:
## 3.1. CUSTOMER MASTER

# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
customer_stage_df = spark.read.format("parquet").option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/customers")


# Write to bronze layer / raw layer
customer_stage_df.coalesce(1).write.mode("overwrite").parquet(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/customers")


# Create a temporary table
spark.read.format("parquet").option("inferschema",True).load(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/customers").createOrReplaceTempView("b_customer_master")


# Run some quick stats
spark.sql(f"select distinct status, count(*) customers from b_customer_master group by status").show(truncate=False)
spark.sql(f"select count(*) customers from b_customer_master").show(truncate=False)
spark.sql(f"select count(distinct *) distinct_customers from b_customer_master").show(truncate=False)
spark.sql(f"select * from b_customer_master limit 2").show(truncate=False)


In [ ]:
## 3.2. Customer master sensitive (allergies)

# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
customer_stage_df = spark.read.format("parquet").option("header", True).option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/customers_sensitive")

# Write to bronze layer / raw layer
customer_stage_df.coalesce(1).write.mode("overwrite").parquet(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/customers_sensitive")

# Create a temporary table
spark.read.format("parquet").option("inferschema",True).load(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/customers_sensitive").createOrReplaceTempView("b_customer_master_sensitive")

# Some quick stats
spark.sql(f"select count(*) customers_with_allergies from b_customer_master_sensitive").show(truncate=False)
spark.sql(f"select count(distinct *) distinct_customers_with_allergies from b_customer_master_sensitive").show(truncate=False)
spark.sql(f"select * from b_customer_master_sensitive limit 2").show(truncate=False)

In [ ]:
## 3.3. product master

# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
product_master_stage_df= spark.read.format("parquet").option("header", True).option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/products")

# Write to bronze layer / raw layer
product_master_stage_df.coalesce(1).write.mode("overwrite").parquet(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/product_master")

# Create a temporary table
spark.read.format("parquet").option("inferschema",True).load(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/product_master").createOrReplaceTempView("b_product_master")

# Some quick stats
spark.sql(f"select count(*) products from b_product_master").show(truncate=False)
spark.sql(f"select * from b_product_master limit 2").show(truncate=False)

In [ ]:
## 3.4. orders

# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
orders_stage_df = spark.read.format("parquet").option("header", True).option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/orders")

# Write to bronze layer / raw layer
orders_stage_df.coalesce(1).write.mode("overwrite").parquet(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/orders")

# Create a temporary table
spark.read.format("parquet").option("inferschema",True).load(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/orders").createOrReplaceTempView("b_orders")

# Some quick stats
spark.sql(f"select count(distinct *) distinct_orders from b_orders").show(truncate=False)
spark.sql(f"select * from b_orders limit 2").show(truncate=False)
spark.sql(f"select count(distinct order_date) distinct_order_dates from b_orders").show(truncate=False)
spark.sql(f"select count(distinct customer_id) distinct_customers from b_orders").show(truncate=False)

In [ ]:
## 3.5. order_items

# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
order_items_stage_df = spark.read.format("parquet").option("header", True).option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/order_items")

# Write to bronze layer / raw layer
order_items_stage_df.coalesce(1).write.mode("overwrite").parquet(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/order_items")

# Create a temporary table
spark.read.format("parquet").option("inferschema",True).load(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/order_items").createOrReplaceTempView("b_order_items")

# Some quick stats
spark.sql(f"select count(*) order_items from b_order_items").show(truncate=False)
spark.sql(f"select count(distinct *) distinct_order_items from b_order_items").show(truncate=False)
spark.sql(f"select * from b_order_items limit 2").show(truncate=False)


In [ ]:
## 3.6. regions

# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
regions_stage_df = spark.read.format("parquet").option("header", True).option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/regions")

# Write to bronze layer / raw layer
regions_stage_df.coalesce(1).write.mode("overwrite").parquet(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/regions")

# Create a temporary table
spark.read.format("parquet").option("inferschema",True).load(f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/regions").createOrReplaceTempView("b_regions")

# Some quick stats
spark.sql(f"select count(*) regions from b_regions").show(truncate=False)
spark.sql(f"select count(distinct *) distinct_regions from b_regions").show(truncate=False)
spark.sql(f"select * from b_regions limit 2").show(truncate=False)

# 4. [Medallion Architecture] Build the curated layer / silver layer

In [ ]:
spark.sql("show tables in froyo_ns").show(truncate=False)

## 4.1. Curate customer data

### 4.1.1. Quick review of data as it exists in the bronze layer

In [ ]:
spark.sql(f"select * from b_customer_master limit 2").show(truncate=False)
spark.sql(f"select * from b_regions limit 2").show(truncate=False)


### 4.1.2. Rules we will apply
1. Deduplicate
2. Explode the json
3. Get the region details into the customer table
4. Ensure we dont have too many small files

In [ ]:
# Coalescing as we have VERY small data for the purpose of a hands on lab. Consider other Spark optimizations for optimal file sizing without skew.

silver_df=spark.sql("select c.customer_id, c.customer_nm,get_json_object(c.demographics, '$.age_bracket') AS age_bracket, " \
                    "get_json_object(c.demographics, '$.income') AS income,r.city, r.state_province as state_cd, " \
                    "r.zip_code as zip_cd, r.country as country_cd,c.status,c.consent_ts " \
                    "from b_customer_master c left outer join b_regions r on c.region_id=r.region_id").dropDuplicates()
silver_df.show(2, truncate=False)
silver_df.count()
silver_df.printSchema()
silver_df.coalesce(1).write.format("iceberg").mode("overwrite").saveAsTable("froyo_ns.s_customer_master")


### 4.1.3. Curate customer sensitive data

We will not touch this yet - we will work with this when fine grained access control and column masking is available with Lakehouse runtime catalog.

In [ ]:
# Coalescing as we have very small data

silver_df=spark.sql("select * from b_customer_master_sensitive").dropDuplicates()
silver_df.show(2, truncate=False)
silver_df.count()
silver_df.printSchema()
silver_df.coalesce(1).write.format("iceberg").mode("overwrite").saveAsTable("froyo_ns.s_customer_master_sensitive")

In [ ]:
spark.sql("show tables in froyo_ns").show(truncate=False)

## 4.2. Curate product master data


### 4.2.1. Curate product master

Rules:
1. Add GCS URI of recipes PDF as a column
2. Deduplicate

In [ ]:
spark.sql("select * from b_product_master limit 5").show(truncate=False)

In [ ]:
silver_df=spark.sql("select DISTINCT product_id, product_name as product_nm, unit_price, "\
                    f"concat(\"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo-recipe-pdfs/\",replace(lower(product_name),' ','_'),'.pdf') as recipe " \
                    "from b_product_master").dropDuplicates()

silver_df.coalesce(1).writeTo("froyo_ns.s_product_master") \
    .tableProperty("write.format.default", "parquet") \
    .tableProperty("write.target-file-size-bytes", "536870912") \
    .createOrReplace()

In [ ]:
spark.sql("select * from froyo_ns.s_product_master limit 5").show(truncate=False)

In [ ]:
spark.sql("DESCRIBE formatted froyo_ns.s_product_master").show(truncate=False)

In [ ]:
spark.sql("show tables in froyo_ns").show(truncate=False)

### 4.2.2. Copy the recipes over to silver layer

In [ ]:
import pandas as pd
from google.cloud import storage
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

# 1. Variables
COMMON_PATH = "froyo-recipe-pdfs"

# 2. Grabbing only valid product slugs
df_files_to_copy = spark.sql(f"""
    SELECT DISTINCT
        replace(lower(product_nm), ' ', '_') as slug
    FROM froyo_ns.s_product_master
    WHERE product_nm IS NOT NULL
""")

df_files_to_copy.show(2, truncate=False)

# 3. We will use mapInPandas - it requires a defined return schema
result_schema = StructType([StructField("status", StringType())])

# 4. Function for parallel copy
def parallel_copy_pdfs(pdf_iterator):
    """Vectorized sync using a single client per batch of product names."""
    client = storage.Client()
    source_bucket = client.bucket(STAGE_BUCKET_NAME)
    dest_bucket = client.bucket(ICEBERG_LAKEHOUSE_BUCKET_NAME)

    for pdf in pdf_iterator:
        for slug in pdf['slug']:
            filename = f"{slug}.pdf"
            full_path = f"{COMMON_PATH}/{filename}"

            source_blob = source_bucket.blob(full_path)

            if source_blob.exists():
                source_bucket.copy_blob(source_blob, dest_bucket, full_path)

        yield pd.DataFrame({'status': ['batch_processed']})

# 5. High-parallelism across Spark executors
# We use repartition to ensure we have enough 'worker threads' hitting GCS.
sync_job = df_files_to_copy.repartition(200).mapInPandas(parallel_copy_pdfs, result_schema)

# Spark Connect requires an action to trigger the mapInPandas logic.
total_batches = sync_job.count()
print(f"Sync complete. Successfully processed {total_batches} parallel batches.")

## 4.3 Curate order data


1. Orders table has data quality issues - same order ID, multiple order dates for frozen yogurts - this needs cleaning up
2. Denomalize order data to include order items
3. To include include product ID
4. Persist with a partitioning scheme of YYYY-MM
5. Clustering by product_id

In [ ]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# 1. Define the window (Grouping by ID, ordering by Date)
window_spec = Window.partitionBy("order_id").orderBy(F.col("order_date").desc())

# 2. Filter for only the 'top' record
deduped_df = spark.table("b_orders") \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .drop("rank")

deduped_df=deduped_df.withColumnRenamed("order_date","order_dt").dropDuplicates()
# Deduped = 299948

deduped_df.writeTo("froyo_ns.s_orders") \
    .partitionedBy(F.months("order_dt")) \
    .tableProperty("write.distribution-mode", "hash") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

In [ ]:
spark.sql(f"select count(distinct order_dt) distinct_order_dates from froyo_ns.s_orders").show(truncate=False)
spark.sql(f"select count(distinct customer_id) distinct_customers from froyo_ns.s_orders").show(truncate=False)
spark.sql(f"select count(order_id) orders from froyo_ns.s_orders").show(truncate=False)

In [ ]:
from pyspark.sql import functions as F
# 1. Enable Adaptive Query Execution (AQE)
spark.conf.set("spark.sql.adaptive.enabled", "true")

# 2. Use Broadcast Hint and move logic into SQL for the optimizer
query = """
SELECT /*+ BROADCAST(p) */
    DISTINCT
    o.order_id,
    CAST(o.order_dt AS DATE) as order_dt,
    o.order_total,
    o.customer_id,
    oi.order_item_id,
    oi.product_name as product_nm,
    oi.quantity,
    oi.unit_price,
    oi.line_total
FROM froyo_ns.s_orders o
JOIN b_order_items oi ON (oi.order_id = o.order_id)
"""

# Create the DF and drop duplicates once
silver_df = spark.sql(query).dropDuplicates()


# Use the "Hash" distribution mode for Iceberg
# This ensures data is pre-shuffled by the partition key (order_dt)
# so you don't create a "Small File" disaster.
silver_df.writeTo("froyo_ns.s_order_history") \
    .partitionedBy(F.months("order_dt")) \
    .tableProperty("write.distribution-mode", "hash") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

In [ ]:
# Add product_id & product_repice columns
spark.sql("alter table froyo_ns.s_order_history add column product_id long,product_recipe string;")

In [ ]:
spark.sql("desc formatted froyo_ns.s_order_history").show(50,truncate=False)

In [ ]:
query = """MERGE INTO froyo_ns.s_order_history AS oh
USING froyo_ns.s_product_master AS pm
ON oh.product_nm = pm.product_nm
WHEN MATCHED THEN
  UPDATE SET oh.product_id = pm.product_id,
  oh.product_recipe=pm.recipe;"""

# Execute the merge
spark.sql(query)

# Take a peek at two records
spark.sql("select * from froyo_ns.s_order_history limit 2").show(truncate=False)

# Quick count
spark.sql("select count(*) from froyo_ns.s_order_history").show(truncate=False)

In [ ]:
spark.sql(f"select count(distinct customer_id) distinct_customers from froyo_ns.s_order_history").show(truncate=False)
spark.sql(f"select count(distinct order_dt) distinct_order_dt from froyo_ns.s_order_history").show(truncate=False)
spark.sql(f"select count(distinct product_id) distinct_product_id from froyo_ns.s_order_history").show(truncate=False)
spark.sql(f"select count(*) record_count from froyo_ns.s_order_history").show(truncate=False)

# 5. [Medallion Architecture] Build the consumption layer / gold layer

### 5.1. Create the gold layer denormalized table

We will denomalize for query efficiency and persist to Iceberg format in the Lakehouse.

In [ ]:
gold_table_sql="""select distinct
oh.order_id,oh.order_dt,oh.order_total,oh.order_item_id,oh.product_id,oh.product_nm,oh.product_recipe, oh.unit_price,oh.quantity,oh.line_total,
c.customer_id,c.customer_nm,c.age_bracket as customer_age_bracket,c.income as customer_income,c.city as customer_city,c.state_cd as customer_state_cd,
c.zip_cd as customer_zip_cd,c.country_cd as customer_country_cd,c.status as customer_status,c.consent_ts as customer_consent_ts
from froyo_ns.s_order_history oh left outer join froyo_ns.s_customer_master c on oh.customer_id=c.customer_id"""

spark.sql(f"""{gold_table_sql} LIMIT 2""").show(truncate=False)

In [ ]:
gold_df=spark.sql(gold_table_sql)

gold_df.writeTo("froyo_ns.g_orders_enriched") \
  .partitionedBy(F.months("order_dt")) \
  .tableProperty("write.distribution-mode", "hash") \
  .tableProperty("write.format.default", "parquet") \
  .createOrReplace()

In [ ]:
gold_df.printSchema()

In [ ]:
gold_df.count()

In [ ]:
# Quick verification
gold_df.show(2, truncate=False)

# 6. [Medallion Architecture] Build the distribution layer / platinum layer -> Reporting Data Mart

The code in these sections was authored using prompts, with assistance from Data Science Agent in Colab notebook.

### 6.1. Revenue trends

In [ ]:
# prompt: Generate a sales revenue repo by month based off of Iceberg table froyo_ns.g_orders_enriched table and persist it to the Iceberg table: froyo_ns.p_rdm_revenue_by_month

import pandas as pd
import pandas_gbq

# Generate sales revenue report by month
sales_revenue_by_month_df = spark.sql("""
    SELECT
        DATE_TRUNC('MONTH', order_dt) AS sales_month,
        SUM(line_total) AS total_revenue
    FROM froyo_ns.g_orders_enriched
    GROUP BY 1
    ORDER BY 1
""")

# Persist to Iceberg table
sales_revenue_by_month_df.writeTo("froyo_ns.p_rdm_revenue_by_month") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

# Display the result (optional)
sales_revenue_by_month_df.show()

In [ ]:
# prompt: Help me visualize with a bar chart, the Spark dataframe from above (considering its small data)

# NOTE: Pyspark code generation is currently in PREVIEW.
import matplotlib.pyplot as plt

# Convert the Spark DataFrame to a Pandas DataFrame
sales_revenue_pd_df = sales_revenue_by_month_df.toPandas()

# Create the bar chart
plt.figure(figsize=(12, 6))
plt.bar(sales_revenue_pd_df['sales_month'], sales_revenue_pd_df['total_revenue'], color='skyblue')
plt.xlabel('Sales Month')
plt.ylabel('Total Revenue')
plt.title('Total Revenue by Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.2. Average Order Value report

In [ ]:
# prompt: Generate average order value report by month based off of Iceberg table froyo_ns.g_orders_enriched table and persist it to the Iceberg table: froyo_ns.p_rdm_averge_order_value

import matplotlib.pyplot as plt
# Generate average order value report by month
average_order_value_df = spark.sql("""
    SELECT
        DATE_TRUNC('MONTH', order_dt) AS sales_month,
        AVG(order_total) AS average_order_value
    FROM froyo_ns.g_orders_enriched
    GROUP BY 1
    ORDER BY 1
""")

# Persist to Iceberg table
average_order_value_df.writeTo("froyo_ns.p_rdm_averge_order_value") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

# Display the result (optional)
average_order_value_df.show()

# Convert the Spark DataFrame to a Pandas DataFrame
average_order_value_pd_df = average_order_value_df.toPandas()

# Create the bar chart
plt.figure(figsize=(12, 6))
plt.bar(average_order_value_pd_df['sales_month'], average_order_value_pd_df['average_order_value'], color='lightcoral')
plt.xlabel('Sales Month')
plt.ylabel('Average Order Value')
plt.title('Average Order Value by Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.3. Top 10 products

In [ ]:
# prompt: Generate top 10 products report by revenue based off of Iceberg table froyo_ns.g_orders_enriched table and persist it to the Iceberg table: froyo_ns.p_rdm_top_ten_products_by_revenue. I want to visualize them as a pie chart

import matplotlib.pyplot as plt

# Generate top 10 products report by revenue
top_10_products_df = spark.sql("""
    SELECT
        product_nm,
        SUM(line_total) AS total_revenue
    FROM froyo_ns.g_orders_enriched
    GROUP BY product_nm
    ORDER BY total_revenue DESC
    LIMIT 10
""")

# Persist to Iceberg table
top_10_products_df.writeTo("froyo_ns.p_rdm_top_ten_products_by_revenue") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

# Display the result (optional)
top_10_products_df.show()

# Convert the Spark DataFrame to a Pandas DataFrame
top_10_products_pd_df = top_10_products_df.toPandas()

# Create the pie chart
plt.figure(figsize=(10, 10))
plt.pie(top_10_products_pd_df['total_revenue'], labels=top_10_products_pd_df['product_nm'], autopct='%1.1f%%', startangle=140)
plt.title('Top 10 Products by Revenue')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.tight_layout()
plt.show()

### 6.4. Customer segmentation

In [ ]:
# prompt: Generate customer segmentation report by customer age bracket based off of Iceberg table froyo_ns.g_orders_enriched table and persist it to the Iceberg table: froyo_ns.p_rdm_customer_segmentation_by_age. I want to visualize with a line chart

import matplotlib.pyplot as plt

# Generate customer segmentation report by customer age bracket
customer_segmentation_df = spark.sql("""
    SELECT
        customer_age_bracket,
        COUNT(DISTINCT customer_id) AS number_of_customers,
        SUM(order_total) AS total_spend
    FROM froyo_ns.g_orders_enriched
    GROUP BY customer_age_bracket
    ORDER BY customer_age_bracket
""")

# Persist to Iceberg table
customer_segmentation_df.writeTo("froyo_ns.p_rdm_customer_segmentation_by_age") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

# Display the result (optional)
customer_segmentation_df.show()

# Convert the Spark DataFrame to a Pandas DataFrame
customer_segmentation_pd_df = customer_segmentation_df.toPandas()

# Create the line chart for number of customers
plt.figure(figsize=(12, 6))
plt.plot(customer_segmentation_pd_df['customer_age_bracket'], customer_segmentation_pd_df['number_of_customers'], marker='o', color='blue')
plt.xlabel('Customer Age Bracket')
plt.ylabel('Number of Customers')
plt.title('Number of Customers by Age Bracket')
plt.grid(True)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Create the line chart for total spend
plt.figure(figsize=(12, 6))
plt.plot(customer_segmentation_pd_df['customer_age_bracket'], customer_segmentation_pd_df['total_spend'], marker='o', color='green')
plt.xlabel('Customer Age Bracket')
plt.ylabel('Total Spend')
plt.title('Total Spend by Customer Age Bracket')
plt.grid(True)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: Generate customer segmentation report by customer age bracket based off of Iceberg table froyo_ns.g_orders_enriched table and persist it to the Iceberg table: froyo_ns.p_rdm_customer_segmentation_by_age. I want to visualize with a line chart

import matplotlib.pyplot as plt

# Generate customer segmentation report by customer age bracket
customer_segmentation_df = spark.sql("""
    SELECT
        customer_age_bracket,
        COUNT(DISTINCT customer_id) AS number_of_customers,
        SUM(order_total) AS total_revenue
    FROM froyo_ns.g_orders_enriched
    GROUP BY customer_age_bracket
    ORDER BY customer_age_bracket
""")

# Persist to Iceberg table
customer_segmentation_df.writeTo("froyo_ns.p_rdm_customer_segmentation_by_age") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

# Display the result (optional)
customer_segmentation_df.show()

# Convert the Spark DataFrame to a Pandas DataFrame
customer_segmentation_pd_df = customer_segmentation_df.toPandas()

# Create the bar chart for number of customers
plt.figure(figsize=(12, 6))
plt.bar(customer_segmentation_pd_df['customer_age_bracket'], customer_segmentation_pd_df['number_of_customers'], color='lightgreen')
plt.xlabel('Customer Age Bracket')
plt.ylabel('Number of Customers')
plt.title('Customer Segmentation by Age Bracket (Number of Customers)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Create the bar chart for total revenue
plt.figure(figsize=(12, 6))
plt.bar(customer_segmentation_pd_df['customer_age_bracket'], customer_segmentation_pd_df['total_revenue'], color='lightsalmon')
plt.xlabel('Customer Age Bracket')
plt.ylabel('Total Revenue')
plt.title('Customer Segmentation by Age Bracket (Total Revenue)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

##

# 7. [Table Format primer] Apache Iceberg basics

This section is Apache Iceberg specific and covers table, metadata inspection, and table maintenance. The content here is inspired by Nikhil Manjunatha's lab: https://github.com/nikhil6790/table-format-lab-iceberg/tree/main

## 7.1. CRUD operations

In [ ]:
spark.sql("SELECT max(sales_month) FROM froyo_ns.p_rdm_revenue_by_month").show()

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.snapshots").show(truncate=False)

### 7.1.1. Insert/Create

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

In [ ]:
spark.sql("""INSERT INTO froyo_ns.p_rdm_revenue_by_month(sales_month, total_revenue)
            values(to_date('2026-04-01 00:00:00'), 423000)
        """).show()

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.snapshots").show(truncate=False)

### 7.1.2. Update

In [ ]:
spark.sql("SELECT * FROM froyo_ns.p_rdm_revenue_by_month WHERE sales_month='2026-03-01 00:00:00'").show()

In [ ]:
spark.sql("""UPDATE froyo_ns.p_rdm_revenue_by_month
            SET total_revenue = 800000
            WHERE sales_month='2026-03-01 00:00:00'
        """).show()

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.snapshots").show(truncate=False)

### 7.1.3. Delete

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

In [ ]:
spark.sql("SELECT * FROM froyo_ns.p_rdm_revenue_by_month WHERE sales_month='2026-04-01 00:00:00'").show()

In [ ]:
spark.sql("""DELETE FROM froyo_ns.p_rdm_revenue_by_month
            where sales_month = '2026-04-01 00:00:00'
        """).show()

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.snapshots").show(truncate=False)

## 7.2. Table inspection



froyo_ns.g_orders_enriched table is partitioned; So let's use the same to review table inspection features in Apache Iceberg

In [ ]:
spark.sql("desc formatted froyo_ns.g_orders_enriched").show(50,truncate=False)

In [ ]:
spark.table("froyo_ns.g_orders_enriched.partitions").show(truncate=False)

In [ ]:
spark.table("froyo_ns.g_orders_enriched.history").show(truncate=False)


In [ ]:
spark.table("froyo_ns.g_orders_enriched.metadata_log_entries").show(truncate=False)

In [ ]:
spark.table("froyo_ns.g_orders_enriched.snapshots").show(truncate=False)

In [ ]:
spark.table("froyo_ns.g_orders_enriched.files").show(truncate=False)

In [ ]:
spark.table("froyo_ns.g_orders_enriched.all_files").show(truncate=False)

In [ ]:
spark.table("froyo_ns.g_orders_enriched.manifests").show(truncate=False)

## 7.3. Time travel



We did a bunch of updates to the table froyo_ns.p_rdm_revenue_by_month. So, lets use this table for reviewing time travel features

In [ ]:
spark.sql(f"select committed_at, snapshot_id, operation from froyo_ns.p_rdm_revenue_by_month.snapshots").show(truncate=False)

In [ ]:
spark.sql(f"select * from froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

Update the snapshot IDs below to reflect your snapshot IDs from the table above

In [ ]:
from pyspark.sql.functions import col

INSERT_SNAPSHOT_ID = spark.sql(f"SELECT snapshot_id FROM \
(SELECT snapshot_id, ROW_NUMBER() OVER(ORDER BY committed_at ASC) rownum FROM froyo_ns.p_rdm_revenue_by_month.snapshots) \
a where a.rownum =2").collect()[0][0]

print("INSERT_SNAPSHOT_ID ", INSERT_SNAPSHOT_ID)

print(f"TABLE STATE AT INSERT_SNAPSHOT_ID OF {INSERT_SNAPSHOT_ID} IS:")
spark.read \
    .option("snapshot-id", INSERT_SNAPSHOT_ID) \
    .table("froyo_ns.p_rdm_revenue_by_month").filter(col('sales_month').isin('2026-03-01 00:00:00','2026-04-01 00:00:00')).sort('sales_month').show(truncate=False)

In [ ]:
UPDATE_SNAPSHOT_ID = spark.sql(f"SELECT snapshot_id FROM \
(SELECT snapshot_id, ROW_NUMBER() OVER(ORDER BY committed_at ASC) rownum FROM froyo_ns.p_rdm_revenue_by_month.snapshots) \
a where a.rownum =3").collect()[0][0]

print("UPDATE_SNAPSHOT_ID ", UPDATE_SNAPSHOT_ID)

print(f"TABLE STATE AT UPDATE_SNAPSHOT_ID OF {UPDATE_SNAPSHOT_ID} IS:")
spark.read \
    .option("snapshot-id", UPDATE_SNAPSHOT_ID) \
    .table("froyo_ns.p_rdm_revenue_by_month").filter(col('sales_month').isin('2026-03-01 00:00:00','2026-04-01 00:00:00')).sort('sales_month').show(truncate=False)

In [ ]:
DELETE_SNAPSHOT_ID = spark.sql(f"SELECT snapshot_id FROM \
(SELECT snapshot_id, ROW_NUMBER() OVER(ORDER BY committed_at ASC) rownum FROM froyo_ns.p_rdm_revenue_by_month.snapshots) \
a where a.rownum =4").collect()[0][0]

print("DELETE_SNAPSHOT_ID ", DELETE_SNAPSHOT_ID)

print(f"TABLE STATE AT DELETE_SNAPSHOT_ID OF {DELETE_SNAPSHOT_ID} IS:")
spark.read \
    .option("snapshot-id", DELETE_SNAPSHOT_ID) \
    .table("froyo_ns.p_rdm_revenue_by_month").filter(col('sales_month').isin('2026-03-01 00:00:00','2026-04-01 00:00:00')).sort('sales_month').show(truncate=False)

## 7.4. Snapshot management

Update the snapshot IDs below to reflect your snapshot IDs from the table above

In [ ]:
spark.sql(f"select * from froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

In [ ]:
#Fetch the 2nd snapshot update for this example
ROLLBACK_SNAPSHOT_ID = spark.sql(f"SELECT snapshot_id FROM \
(SELECT snapshot_id, ROW_NUMBER() OVER(ORDER BY committed_at ASC) rownum FROM froyo_ns.p_rdm_revenue_by_month.snapshots) \
a where a.rownum =2").collect()[0][0]

print("ROLLBACK_SNAPSHOT_ID ", ROLLBACK_SNAPSHOT_ID)

In [ ]:
# Query the table before rollback - note it has only 2026-03-01 data
spark.sql("select * from froyo_ns.p_rdm_revenue_by_month where sales_month in ('2026-03-01','2026-04-01')").show(truncate=False)

In [ ]:
# The snapshot below has an extra record that we inserted
print(f"TABLE STATE AT snapshot-id '{ROLLBACK_SNAPSHOT_ID}'")
spark.read \
    .option("snapshot-id", f'{ROLLBACK_SNAPSHOT_ID}') \
    .table("froyo_ns.p_rdm_revenue_by_month").filter(col('sales_month').isin('2026-03-01 00:00:00','2026-04-01 00:00:00')).show(truncate=False)

In [ ]:
# Build snapshot rollback statement to have data for the month of April 2026; Execute the rollback
SNAPSHOT_ROLLBACK_STATEMENT = f"CALL froyo_iceberg_catalog.system.rollback_to_snapshot('froyo_ns.p_rdm_revenue_by_month',{ROLLBACK_SNAPSHOT_ID})"
print(SNAPSHOT_ROLLBACK_STATEMENT)

spark.sql(f"{SNAPSHOT_ROLLBACK_STATEMENT}").show()

In [ ]:
# Query the table post the rollback
spark.sql("select * from froyo_ns.p_rdm_revenue_by_month where sales_month in ('2026-03-01','2026-04-01')").show(truncate=False)


In [ ]:
# Now that we know how rollback works, lets reset the table now that we know how rollbacks work
spark.sql("DELETE FROM froyo_ns.p_rdm_revenue_by_month where sales_month='2026-04-01'")

## 7.5. Table maintenance

In [ ]:
TABLE_NAME="p_rdm_revenue_by_month"
ICEBERG_NAMESPACE="froyo_ns"

#fully qualified table name
FQTN=f"{ICEBERG_NAMESPACE}.{TABLE_NAME}"

print("Fully quailified table name :",FQTN)

### 7.5.1. EXPIRE SNAPSHOTS


**Get base file counts from the table folder**

In [ ]:
DATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/data/*.parquet | wc -l
print("DATA_FILE_COUNT",DATA_FILE_COUNT)

METADATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*.json | wc -l
print("METADATA_FILE_COUNT",METADATA_FILE_COUNT)

MANIFEST_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*m[0-9].avro | wc -l
print("MANIFEST_FILE_COUNT",MANIFEST_FILE_COUNT)

MANIFEST_LIST_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/snap*.avro | wc -l
print("MANIFEST_LIST_COUNT",MANIFEST_LIST_COUNT)

In [ ]:
spark.sql(f"select * from froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

In [ ]:
EXP_TS = spark.sql(f"select committed_at from (SELECT committed_at, ROW_NUMBER() OVER(ORDER BY committed_at ASC) rownum from {FQTN}.snapshots) a where a.rownum =3").collect()[0][0]
print("Expiration Timestamp", EXP_TS)

In [ ]:
# Expire snapshots older than the timestamp above
#spark.sql(f"CALL froyo_catalog.system.expire_snapshots(table => '{FQTN}',older_than=> TIMESTAMP '{EXP_TS}', retain_last => 2)").show(truncate=False)
spark.sql(f"CALL froyo_iceberg_catalog.system.expire_snapshots(table => '{FQTN}',older_than=> TIMESTAMP '{EXP_TS}')").show(truncate=False)

In [ ]:
spark.sql(f"select * from froyo_ns.p_rdm_revenue_by_month.history").show(truncate=False)

Get file counts after snapshot expiration

In [ ]:
DATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/data/*.parquet | wc -l
print("DATA_FILE_COUNT",DATA_FILE_COUNT)

METADATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*.json | wc -l
print("METADATA_FILE_COUNT",METADATA_FILE_COUNT)

MANIFEST_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*m[0-9].avro | wc -l
print("MANIFEST_FILE_COUNT",MANIFEST_FILE_COUNT)

MANIFEST_LIST_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/snap*.avro | wc -l
print("MANIFEST_LIST_COUNT",MANIFEST_LIST_COUNT)

# NOTE:

# Iceberg identifies the snapshots it can delete safely and deletes the corresponding data files, manifests and manifest lists
# Also a new metadata file is added and this file will not maintain information about the expired snapshots so they are no
# longer available for time travel queries.

In [ ]:
spark.table("froyo_ns.p_rdm_revenue_by_month.snapshots").show(truncate=False)

### 7.5.2. REWRITE MANIFESTS

In [ ]:
# Rewrite manifests for a table to optimize scan planning.
spark.sql("CALL froyo_iceberg_catalog.system.rewrite_manifests('froyo_ns.p_rdm_revenue_by_month')").show(truncate=False)

**Get file counts from the table folder after rewriting manifests**


In [ ]:
DATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/data/*.parquet | wc -l
print("DATA_FILE_COUNT",DATA_FILE_COUNT)

METADATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*.json | wc -l
print("METADATA_FILE_COUNT",METADATA_FILE_COUNT)

MANIFEST_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*m[0-9].avro | wc -l
print("MANIFEST_FILE_COUNT",MANIFEST_FILE_COUNT)

MANIFEST_LIST_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/snap*.avro | wc -l
print("MANIFEST_LIST_COUNT",MANIFEST_LIST_COUNT)


# NOTE:

# Rewriting manifests performs below operations
# i. Align manifest files with table partitioning
# ii. Sort data files in manifest based on partition spec fields
# iii. Optimize scan planning

# Adds new snapshot and manifest list to indicate changes to manifests



### 7.5.3. REWRITE DATA FILES

NOTE:

1. If the data files are already compacted then rewriting data files does not impact the files
2. There are different compaction strategies, default is binpacking if none specified.

In [ ]:
# Rewrite data file using lexicographical sort order compaction strategy
spark.sql(f"CALL froyo_iceberg_catalog.system.rewrite_data_files(table => 'froyo_ns.s_customer_master',strategy => 'sort', sort_order => 'customer_nm ASC NULLS LAST' )").show(truncate=False)

In [ ]:
# Rewrite data file using zorder sort compaction strategy
spark.sql(f"CALL froyo_iceberg_catalog.system.rewrite_data_files(table => 'froyo_ns.g_orders_enriched',strategy => 'sort', sort_order => 'zorder(customer_state_cd,customer_city)')").show(truncate=False)

### 7.5.4. CLEAR OLD METADATA FILES

In [ ]:
DATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/data/*.parquet | wc -l
print("DATA_FILE_COUNT",DATA_FILE_COUNT)

METADATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*.json | wc -l
print("METADATA_FILE_COUNT",METADATA_FILE_COUNT)

MANIFEST_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*m[0-9].avro | wc -l
print("MANIFEST_FILE_COUNT",MANIFEST_FILE_COUNT)

MANIFEST_LIST_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/snap*.avro | wc -l
print("MANIFEST_LIST_COUNT",MANIFEST_LIST_COUNT)


In [ ]:
#Set auto metadata cleanup to true
spark.sql(f'ALTER TABLE {FQTN} SET TBLPROPERTIES("write.metadata.delete-after-commit.enabled"=true)').show(truncate=False)

#Set max versions of metadata files to be retained
spark.sql(f'ALTER TABLE {FQTN} SET TBLPROPERTIES("write.metadata.previous-versions-max"=3)').show(truncate=False)

In [ ]:
DATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/data/*.parquet | wc -l
print("DATA_FILE_COUNT",DATA_FILE_COUNT)

METADATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*.json | wc -l
print("METADATA_FILE_COUNT",METADATA_FILE_COUNT)

MANIFEST_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*m[0-9].avro | wc -l
print("MANIFEST_FILE_COUNT",MANIFEST_FILE_COUNT)

MANIFEST_LIST_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/snap*.avro | wc -l
print("MANIFEST_LIST_COUNT",MANIFEST_LIST_COUNT)

# NOTE:
# Iceberg clears all older metadata files and retains only specified 'n' previous versions (3 in our case)
# It adds a new metadata file to commit the transaction of clearing older metadata files, hence we have n+1 (4 in our case) metadata files


### 7.5.5. REMOVE ORPHAN FILES

Orphan files: Unreferenced files left behind, usually caused by failed writes, uncommitted data, or retained files after a snapshot expiration.

In [ ]:
# Clearing any orphaned (untracked) data files from the data folder
spark.sql(f"CALL froyo_iceberg_catalog.system.remove_orphan_files('{FQTN}')").show(truncate=False)

In [ ]:
DATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/data/*.parquet | wc -l
print("DATA_FILE_COUNT",DATA_FILE_COUNT)

METADATA_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*.json | wc -l
print("METADATA_FILE_COUNT",METADATA_FILE_COUNT)

MANIFEST_FILE_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/*m[0-9].avro | wc -l
print("MANIFEST_FILE_COUNT",MANIFEST_FILE_COUNT)

MANIFEST_LIST_COUNT= !gsutil ls -r gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}/froyo_ns/{TABLE_NAME}/metadata/snap*.avro | wc -l
print("MANIFEST_LIST_COUNT",MANIFEST_LIST_COUNT)

# NOTE: If the procedure finds any orphaned files it will clear them and print the file location of deleted files

# 8. [Optional] Data analysis with Data Science Agent in Colab notebooks using just prompts

In [ ]:
# prompt: Show me revenue trends by month of the year from the Iceberg table froyo_ns.g_orders_enriched. Render as a chart

import matplotlib.pyplot as plt

# Generate sales revenue report by month
sales_revenue_by_month_df = spark.sql("""
    SELECT
        DATE_TRUNC('MONTH', order_dt) AS sales_month,
        SUM(line_total) AS total_revenue
    FROM froyo_ns.g_orders_enriched
    GROUP BY 1
    ORDER BY 1
""")

# Convert the Spark DataFrame to a Pandas DataFrame
sales_revenue_pd_df = sales_revenue_by_month_df.toPandas()

# Create the bar chart
plt.figure(figsize=(12, 6))
plt.bar(sales_revenue_pd_df['sales_month'], sales_revenue_pd_df['total_revenue'], color='skyblue')
plt.xlabel('Sales Month')
plt.ylabel('Total Revenue')
plt.title('Total Revenue by Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: Generate Average Order Value (AOV) by month from the Iceberg table froyo_ns.g_orders_enriched. Render as chart

import matplotlib.pyplot as plt

# Generate average order value report by month
average_order_value_df = spark.sql("""
    SELECT
        DATE_TRUNC('MONTH', order_dt) AS sales_month,
        AVG(order_total) AS average_order_value
    FROM froyo_ns.g_orders_enriched
    GROUP BY 1
    ORDER BY 1
""")

# Convert the Spark DataFrame to a Pandas DataFrame
average_order_value_pd_df = average_order_value_df.toPandas()

# Create the bar chart
plt.figure(figsize=(12, 6))
plt.bar(average_order_value_pd_df['sales_month'], average_order_value_pd_df['average_order_value'], color='lightcoral')
plt.xlabel('Sales Month')
plt.ylabel('Average Order Value')
plt.title('Average Order Value by Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: Show me Top 10 Products by Revenue from the Iceberg table froyo_ns.g_orders_enriched & render as a bar chart

import matplotlib.pyplot as plt

# Generate top 10 products report by revenue
top_10_products_df = spark.sql("""
    SELECT
        product_nm,
        SUM(line_total) AS total_revenue
    FROM froyo_ns.g_orders_enriched
    GROUP BY product_nm
    ORDER BY total_revenue DESC
    LIMIT 10
""")

# Convert the Spark DataFrame to a Pandas DataFrame
top_10_products_pd_df = top_10_products_df.toPandas()

# Create the bar chart
plt.figure(figsize=(12, 6))
plt.bar(top_10_products_pd_df['product_nm'], top_10_products_pd_df['total_revenue'], color='lightgreen')
plt.xlabel('Product Name')
plt.ylabel('Total Revenue')
plt.title('Top 10 Products by Revenue')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: Show me Sales by Age Bracket from the Iceberg table froyo_ns.g_orders_enriched - render as pie chart

import matplotlib.pyplot as plt

# Generate sales by age bracket
sales_by_age_bracket_df = spark.sql("""
    SELECT
        customer_age_bracket,
        SUM(line_total) AS total_sales
    FROM froyo_ns.g_orders_enriched
    GROUP BY customer_age_bracket
    ORDER BY total_sales DESC
""")

# Convert the Spark DataFrame to a Pandas DataFrame
sales_by_age_bracket_pd_df = sales_by_age_bracket_df.toPandas()

# Create the pie chart
plt.figure(figsize=(10, 10))
plt.pie(sales_by_age_bracket_pd_df['total_sales'], labels=sales_by_age_bracket_pd_df['customer_age_bracket'], autopct='%1.1f%%', startangle=140)
plt.title('Sales by Age Bracket')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.tight_layout()
plt.show()

In [ ]:
# prompt: Analyze customer_income vs. order_total from the Iceberg table froyo_ns.g_orders_enriched

import matplotlib.pyplot as plt

# Generate customer income vs. order total report
customer_income_order_total_df = spark.sql("""
    SELECT
        customer_income,
        AVG(order_total) AS average_order_total,
        COUNT(DISTINCT customer_id) AS number_of_customers
    FROM froyo_ns.g_orders_enriched
    GROUP BY customer_income
    ORDER BY customer_income
""")

# Convert the Spark DataFrame to a Pandas DataFrame
customer_income_order_total_pd_df = customer_income_order_total_df.toPandas()

# Create a bar chart for average order total by income bracket
plt.figure(figsize=(12, 6))
plt.bar(customer_income_order_total_pd_df['customer_income'], customer_income_order_total_pd_df['average_order_total'], color='purple')
plt.xlabel('Customer Income Bracket')
plt.ylabel('Average Order Total')
plt.title('Average Order Total by Customer Income Bracket')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Create a bar chart for number of customers by income bracket
plt.figure(figsize=(12, 6))
plt.bar(customer_income_order_total_pd_df['customer_income'], customer_income_order_total_pd_df['number_of_customers'], color='orange')
plt.xlabel('Customer Income Bracket')
plt.ylabel('Number of Customers')
plt.title('Number of Customers by Income Bracket')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: Analyze the distribution of customer_status (e.g., Active vs. Churned) to see which segments are most profitable from the Iceberg table froyo_ns.g_orders_enriched

import matplotlib.pyplot as plt

# Generate customer status vs. total revenue report
customer_status_revenue_df = spark.sql("""
    SELECT
        customer_status,
        SUM(order_total) AS total_revenue
    FROM froyo_ns.g_orders_enriched
    GROUP BY customer_status
    ORDER BY total_revenue DESC
""")

# Convert the Spark DataFrame to a Pandas DataFrame
customer_status_revenue_pd_df = customer_status_revenue_df.toPandas()

# Create a bar chart for total revenue by customer status
plt.figure(figsize=(10, 6))
plt.bar(customer_status_revenue_pd_df['customer_status'], customer_status_revenue_pd_df['total_revenue'], color=['skyblue', 'lightcoral'])
plt.xlabel('Customer Status')
plt.ylabel('Total Revenue')
plt.title('Total Revenue by Customer Status')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Generate customer status vs. number of customers report
customer_status_count_df = spark.sql("""
    SELECT
        customer_status,
        COUNT(DISTINCT customer_id) AS number_of_customers
    FROM froyo_ns.g_orders_enriched
    GROUP BY customer_status
    ORDER BY number_of_customers DESC
""")

# Convert the Spark DataFrame to a Pandas DataFrame
customer_status_count_pd_df = customer_status_count_df.toPandas()

# Create a bar chart for number of customers by customer status
plt.figure(figsize=(10, 6))
plt.bar(customer_status_count_pd_df['customer_status'], customer_status_count_pd_df['number_of_customers'], color=['lightgreen', 'orange'])
plt.xlabel('Customer Status')
plt.ylabel('Number of Customers')
plt.title('Number of Customers by Customer Status')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: Show me heatmap by State/City from  the Iceberg table froyo_ns.g_orders_enriched

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Generate sales data by state and city
sales_by_state_city_df = spark.sql("""
    SELECT
        customer_state_cd,
        customer_city,
        SUM(line_total) AS total_sales
    FROM froyo_ns.g_orders_enriched
    GROUP BY customer_state_cd, customer_city
    ORDER BY customer_state_cd, customer_city
""")

# Convert the Spark DataFrame to a Pandas DataFrame
sales_by_state_city_pd_df = sales_by_state_city_df.toPandas()

# Pivot the table to create a matrix suitable for a heatmap
heatmap_data = sales_by_state_city_pd_df.pivot_table(
    index='customer_state_cd',
    columns='customer_city',
    values='total_sales'
).fillna(0)

# Create the heatmap
plt.figure(figsize=(16, 10))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=.5)
plt.title('Total Sales by State and City')
plt.xlabel('Customer City')
plt.ylabel('Customer State Code')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: I want to analyze zip code penetration from the Iceberg table froyo_ns.g_orders_enriched

import pandas as pd
import matplotlib.pyplot as plt

# Generate zip code penetration report
zip_code_penetration_df = spark.sql("""
    SELECT
        customer_zip_cd,
        COUNT(DISTINCT customer_id) AS number_of_customers,
        SUM(order_total) AS total_revenue
    FROM froyo_ns.g_orders_enriched
    GROUP BY customer_zip_cd
    ORDER BY number_of_customers DESC
    LIMIT 20 -- Limiting to top 20 for better visualization
""")

# Convert the Spark DataFrame to a Pandas DataFrame
zip_code_penetration_pd_df = zip_code_penetration_df.toPandas()

# Create a bar chart for number of customers by zip code
plt.figure(figsize=(14, 7))
plt.bar(zip_code_penetration_pd_df['customer_zip_cd'], zip_code_penetration_pd_df['number_of_customers'], color='teal')
plt.xlabel('Customer Zip Code')
plt.ylabel('Number of Customers')
plt.title('Top 20 Zip Codes by Number of Customers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Create a bar chart for total revenue by zip code
plt.figure(figsize=(14, 7))
plt.bar(zip_code_penetration_pd_df['customer_zip_cd'], zip_code_penetration_pd_df['total_revenue'], color='purple')
plt.xlabel('Customer Zip Code')
plt.ylabel('Total Revenue')
plt.title('Top 20 Zip Codes by Total Revenue')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# prompt: Show me a pie chart with the sales of the top five products from the data in the table from the  Iceberg table froyo_ns.g_orders_enriched

import matplotlib.pyplot as plt

# Generate top 5 products report by revenue
top_5_products_df = spark.sql("""
    SELECT
        product_nm,
        SUM(line_total) AS total_revenue
    FROM froyo_ns.g_orders_enriched
    GROUP BY product_nm
    ORDER BY total_revenue DESC
    LIMIT 5
""")

# Convert the Spark DataFrame to a Pandas DataFrame
top_5_products_pd_df = top_5_products_df.toPandas()

# Create the pie chart
plt.figure(figsize=(10, 10))
plt.pie(top_5_products_pd_df['total_revenue'], labels=top_5_products_pd_df['product_nm'], autopct='%1.1f%%', startangle=140)
plt.title('Top 5 Products by Revenue')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.tight_layout()
plt.show()

## This concludes the tutorial. Proceed back to the lab manual.